# 📝 博客发布助手

这个 Notebook 帮你完成三件事：

1. **发布新文章** —— 自己写 Markdown → 一键 `git push` 上线
2. **Markdown 速查** —— 完整语法对照表，照着写就行
3. **互动功能原型** —— 在 Notebook 里直接体验「选择题调查」和「聊天区」长什么样，思考怎么落地到博客

> 技术栈：Python（标准库 + ipywidgets），博客是 **Hexo + GitHub Pages**，目录在 `C:\Users\Operator\Desktop\blog`


## 0. 激活环境（重要！）

本 Notebook 用 **博客专属环境** `blog`，运行前请确认：

- 右上角 Kernel 选择 **`Blog (Python 3.13)`**（即 conda 的 `blog` 环境）
- 如果没看到，点击内核选择器 → 选「Blog (Python 3.13)」

```bash
# 在终端确认 blog 环境
conda activate blog
python --version   # 应显示 Python 3.13.x
```

> 为什么用 `blog` 环境？因为它和博客一体：既有 **Python 3.13**（跑本 Notebook 的转换/发布逻辑），又有 **Node.js 26**（跑 Hexo 构建），发布流程全在一个环境里搞定，不污染你的 `STU` 学习环境。
> 本 Notebook 只用标准库 + ipywidgets，不需要额外安装任何包。


In [ ]:
# 1. 导入所需库 + 配置博客路径
# 只用标准库，任何 Python 3 都能跑；ipywidgets 是 Notebook 自带的交互控件

import os            # 处理文件路径
import re            # 正则：用于纯文字转 Markdown 时的规则识别
import datetime      # 生成文章日期
import subprocess    # 执行 git 命令（提交推送）
from IPython.display import display, Markdown   # 在 Notebook 里渲染 Markdown 预览

# 博客根目录（如果博客换位置了，改这一行即可）
BLOG_DIR  = r"C:\Users\Operator\Desktop\blog"
POSTS_DIR = os.path.join(BLOG_DIR, "source", "_posts")   # 文章存放目录

print("✅ 博客路径：", BLOG_DIR)
print("✅ 文章目录：", POSTS_DIR, "存在：", os.path.isdir(POSTS_DIR))

## 3. Markdown 格式速查（自己写，别怕）

博客文章就是一个 `.md` 文件，格式规则就下面这些，10 分钟学会 👇

### 📐 标题
```markdown
# 一级标题（文章大标题，一般不手动写）
## 二级标题（小节标题，最常用）
### 三级标题（小节下的小点）
```

### 📝 段落与换行
```markdown
段落之间用空行隔开，会自动分段。

这是第二段。
```
> ⚠️ 同一段内直接回车只是软换行，**想分段必须加空行**。

### 🖊 加粗 / 斜体 / 行内代码
```markdown
**加粗文字**
*斜体文字*
`行内代码`
```

### 📋 列表
```markdown
- 无序列表项
- 第二项

1. 有序列表项
2. 第二项
```

### 🔗 链接与图片
```markdown
[文字链接](https://example.com)
![图片说明](/images/图.jpg)   # 图片放博客的 source/images/ 目录
```

### 💻 代码块（三个反引号包裹）
```markdown
​```python
print("hello")
​```
```

### 📊 表格
```markdown
| 列1 | 列2 |
|-----|-----|
| A   | B   |
```

### 💬 引用与分割线
```markdown
> 引用别人的话

---
```

### ⚠️ 文章头部（front matter）别删
每篇文章开头必须有这段"元信息"，保存时会自动生成（也可手动改）：

```yaml
---
title: 文章标题
date: 2026-08-07 12:00:00
tags:
  - 技术
categories:
  - 技术
---
```

💡 写完正文后，运行下面「保存 + 发布」单元格即可上线。


In [ ]:
# 5. 保存为博客文章 + 一键发布
# 把你写好的 Markdown 存进 source/_posts/，再 git 提交推送 → GitHub Actions 自动部署

def save_post(title: str, content: str) -> str:
    """
    保存为一篇 Hexo 文章（自动加 front matter 模板）
    - title:   文章标题
    - content: Markdown 正文（你自己写的）
    - 返回：保存的文件路径
    """
    today = datetime.date.today().isoformat()
    # front matter 是 Hexo 认识文章"元信息"的头部（标题/日期/分类/标签）
    front = (
        "---\n"
        f"title: {title}\n"
        f"date: {today} 12:00:00\n"
        "tags:\n"
        "  - 技术\n"
        "categories:\n"
        "  - 技术\n"
        "---\n\n"
    )
    safe = title.replace("/", "-").replace("\\", "-").replace(":", "：")
    path = os.path.join(POSTS_DIR, f"{safe}.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write(front + content)
    print(f"✅ 文章已保存：{path}")
    return path


def git_publish(title: str):
    """
    提交并推送博客改动（GitHub Actions 会自动构建部署）
    """
    print("🔄 git add / commit / push …")
    # 依次执行三条 git 命令
    cmds = [
        ['git', '-C', BLOG_DIR, 'add', '.'],
        ['git', '-C', BLOG_DIR, 'commit', '-m', f'新文章：{title}'],
        ['git', '-C', BLOG_DIR, 'push'],
    ]
    for cmd in cmds:
        r = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
        out = (r.stdout or "").strip()
        err = (r.stderr or "").strip()
        if out: print(out)
        if r.returncode != 0 and "error" in err.lower(): print(err)
    print("🎉 已推送！1-2 分钟后访问 https://gutianshuo.github.io")


# ── 用法：先保存，再发布 ──────────────
# 1. 自己写一段 Markdown（参考上面的「Markdown 速查」）
# my_md = "## 我的第一节\n\n这是正文，支持 **加粗**、- 列表 等语法…"
#    （也可以直接用 VS Code 在 source/_posts/ 里写 .md）
# 2. 保存（自动加 front matter）→ 推送上线
# save_post("我的新文章", my_md)
# git_publish("我的新文章")


In [ ]:
# 6. 选择题调查原型（ipywidgets）
# 先在这里体验"单选投票"交互，思考怎么落地到博客文章里

import ipywidgets as widgets

# 三个控件：单选按钮 + 提交按钮 + 输出区
question = widgets.RadioButtons(
    options=["很棒 👍", "一般般", "有待改进 👎"],
    description="评价博客：",
    layout=widgets.Layout(width="60%"),
)
submit = widgets.Button(description="提交投票", button_style="primary")
result = widgets.Output()   # 显示反馈的地方

def on_submit(b):
    """点击提交按钮时触发"""
    with result:
        result.clear_output()                      # 清空旧输出
        print(f"✅ 收到！你选了「{question.value}」，谢谢参与～")
        # 真实博客里：这里会把答案发给后端统计（见下面第 8 节的讨论）

submit.on_click(on_submit)                          # 绑定点击事件

# 把控件排列显示出来
display(widgets.VBox([question, submit, result]))

In [ ]:
# 7. 模拟聊天区原型（Widgets）
# 先在这里体验"对话界面"，思考怎么给博客加评论/聊天区

msg_box = widgets.Textarea(
    placeholder="输入你的留言…",
    description="留言：",
    layout=widgets.Layout(width="70%", height="60px"),
)
send_btn = widgets.Button(description="发送", button_style="success")
chat_area = widgets.Output()      # 聊天记录显示区
history = []                      # 保存所有消息

def on_send(b):
    """点击发送：把消息追加到聊天区"""
    text = msg_box.value.strip()
    if not text:
        return                     # 空消息不发送
    history.append(f"你：{text}")
    with chat_area:
        chat_area.clear_output()
        for line in history:       # 逐条显示历史
            print(line)
    msg_box.value = ""             # 清空输入框

send_btn.on_click(on_send)

display(widgets.VBox([msg_box, send_btn, chat_area]))

## 8. 思考：这两个功能怎么落地到博客？

刚才在 Notebook 里体验的交互（投票、聊天）用的是 **Python 内核**，而你的博客是**纯静态网页**（没有服务器），所以不能直接搬过去。以下是可行方案：

### 📊 选择题调查 / 投票

| 方案 | 做法 | 优点 | 缺点 |
|---|---|---|---|
| **Google Forms 嵌入** | 建一个问卷 → 复制嵌入代码贴进文章 | 免费、结果自动汇总成图表 | 国内访问可能不稳 |
| **腾讯问卷** | 同上，国内版 | 国内访问快 | 界面稍重 |
| **Giscus 表情反应** | 评论系统自带的 👍 反应 | 零配置、轻量 | 只能👍👎，不能自定义选项 |
| **自写静态 JS 投票** | 在文章里加一段 `<script>` 单选按钮 | 完全自主、无第三方 | **结果不持久**（刷新即清零，除非接后端） |

> 💡 **推荐**：单题投票用 **Giscus 的 👍/👎 反应**（免费、无后端、数据存 GitHub）；认真做调研用 **腾讯问卷/Google Forms 嵌入**（结果可汇总）。想 100% 自控但接受"纯前端计数"就用自写 JS。

### 💬 聊天区 / 评论区

| 方案 | 原理 | 优点 | 缺点 |
|---|---|---|---|
| **Giscus** ⭐ | 评论存到 **GitHub Discussions** | 免费、无后端、无审查、国内可访问、支持反应和 Markdown | 读者要有 GitHub 账号 |
| **Utterances** | 评论存到 GitHub Issues | 同上，更老牌 | 功能比 Giscus 少 |
| **Valine / Waline** | 需要 LeanCloud / 自托管后端 | 无账号也能评 | 要配后端、有免费额度限制 |
| **Disqus** | 第三方云服务 | 功能全 | 国内访问差、有广告 |

> 💡 **强烈推荐 Giscus**：和你"不想被审查"的需求完美契合——评论数据存你自己的 GitHub 仓库（Discussions），完全自主；只需在仓库设置里启用 Discussions + 安装 Giscus 应用，然后把一段 `<script>` 贴进主题模板即可。**你的 landscape 主题已经内置了 disqus 和 valine，加 Giscus 也很容易。**

### 落地步骤（如果要做 Giscus）
1. 在 `GuTianshuo.github.io` 仓库 → Settings → 启用 **Discussions**
2. 访问 giscus.app → 按引导生成 `<script>` 代码
3. 把代码贴进主题 `layout/_partial/article.ejs`（评论位置）
4. 推送上线，文章底部就会出现评论区 🎉


## 9. 完整使用流程总结

### 日常发布一篇文章
① 运行「导入配置」cell，选好 blog 内核
② 照着「Markdown 速查」自己写正文（用 VS Code 或本 Notebook 都可以）
③ 运行「保存 + 发布」cell：`save_post("标题", 你的md)` → `git_publish("标题")`
④ 1-2 分钟后访问 https://gutianshuo.github.io 刷新查看

### 小贴士
- **写 Markdown 没把握**？把内容粘贴给 Copilot 说「帮我转成 Markdown」，AI 帮你排版
- **评论区**：落地博客用的是 Giscus（见第 8 节讨论）

### 本 Notebook 用到的关键知识点
| 知识点 | 在哪 |
|---|---|
| conda 环境与 kernel | cell 0 |
| Markdown 语法 | cell 3（速查） |
| Hexo front matter 与 git 自动部署 | cell 5 |
| ipywidgets 交互控件 | cell 6 / 7 |
| 静态博客的功能边界（无后端） | cell 8 |
